# Sayısal Görüntü İşleme Midterm Ödevi: Fluo-N2DH-GOWT1 Hücre Sayımı

Öğrenci numarası: 244329048. Seed = 29048 + 2026 = 31074.

Bu not defteri ödevdeki bütün adımları sırasıyla yeniden üretir. Veri seti `data/Fluo-N2DH-GOWT1/` klasöründe bulunmalıdır (bkz. README.md).

**Not (Bölüm 18.5):** Veri seti görüntülerinden üretilen şekiller GitHub'a yüklenmez. Not defterinin repodaki sürümünde gömülü görüntü çıktıları temizlenmiştir. Şekiller not defteri çalıştırıldığında `figures/` klasörüne yeniden üretilir ve raporda sunulmuştur.

In [1]:
import sys, json
sys.path.insert(0, 'src')
import numpy as np, pandas as pd
import cv2, skimage, scipy, matplotlib
from IPython.display import Image, display
D = 'data/Fluo-N2DH-GOWT1'
print('OpenCV', cv2.__version__, '| scikit-image', skimage.__version__, '| SciPy', scipy.__version__,
      '| NumPy', np.__version__, '| pandas', pd.__version__, '| Matplotlib', matplotlib.__version__)

OpenCV 5.0.0 | scikit-image 0.26.0 | SciPy 1.18.1 | NumPy 2.5.3 | pandas 3.0.5 | Matplotlib 3.11.2


## 1-2. Veri seti inceleme

İki sekansın kare sayısı, görüntü boyutu, veri tipi ve yoğunluk istatistikleri hesaplanır. Bu aşamada referans etiketleri kullanılmaz.

In [2]:
import glob
rows = []
for s in ['01', '02']:
    fs = sorted(glob.glob(f'{D}/{s}/t*.tif'))
    ims = [cv2.imread(f, cv2.IMREAD_UNCHANGED) for f in fs]
    a = np.stack(ims).astype(float)
    rows.append(dict(sequence=s, n_frames=len(fs), shape=ims[0].shape, dtype=str(ims[0].dtype),
                     min=a.min(), max=a.max(), mean_range=(round(a.mean((1,2)).min(),2), round(a.mean((1,2)).max(),2)),
                     p99_mean=round(float(np.percentile(a.reshape(len(fs),-1), 99, axis=1).mean()),1)))
pd.DataFrame(rows)

,sequence,n_frames,shape,dtype,min,max,mean_range,p99_mean
0,01,92,"(1024, 1024)",uint8,0.0,255.0,"(3.63, 4.25)",67.4
1,02,92,"(1024, 1024)",uint8,0.0,182.0,"(1.72, 2.3)",31.7


In [3]:
display(Image('figures/fig0_data_inspection.png'))

## 3. Seed ile görüntü seçimi

"En az 5 karelik boşluk", iki kare arasında en az 5 kare bulunması (kare numarası farkı ≥ 6) olarak uygulanmıştır. Koşullar sağlanmadığında aynı RNG ile yeniden çekim yapılır; deneme sayıları kaydedilir.

In [4]:
from data_selection import select_frames, SEED
sel = select_frames(D)
sel.to_csv('selected_frames.csv', index=False)
print('Seed:', SEED)
sel.groupby('group')['frame'].apply(list)

Seed: 31074


group
development               [0, 2, 29, 32, 44, 52, 54, 55, 81, 87]
test           [17, 18, 19, 20, 21, 35, 36, 37, 38, 39, 72, 7...
validation                                  [12, 21, 62, 70, 72]
Name: frame, dtype: object

## 4. Referans hücre sayıları (geliştirme ve doğrulama)

TRA etiket görüntüsünde 0 dışındaki farklı hücre kimlikleri sayılır. Test referansları kilitli tahminler kaydedilmeden okunamaz.

In [5]:
from references import reference_counts
sel = pd.read_csv('selected_frames.csv', dtype={'sequence': str})
refs = pd.concat([reference_counts(D, sel, 'development'), reference_counts(D, sel, 'validation')])
refs[['group','sequence','frame','filename','reference_count']].to_csv('dev_val_references.csv', index=False)
refs[['group','frame','reference_count']]

,group,frame,reference_count
0,development,0,23
1,development,2,23
2,development,29,24
3,development,32,24
4,development,44,24
5,development,52,21
6,development,54,21
7,development,55,21
8,development,81,20
9,development,87,21


In [6]:
try:
    reference_counts(D, sel, 'test')
except PermissionError as e:
    print('Koruma çalışıyor:', e)

Koruma çalışıyor: Test referansları, test_predictions_locked.csv kaydedilmeden okunamaz.


In [7]:
for f in ['figures/fig_ref_check_dev.png', 'figures/fig_ref_check_small.png', 'figures/fig_ref_check_remaining.png']:
    display(Image(f))

## 5. Baseline

Gri seviye → Otsu → 8-komşuluklu bağlı bileşen analizi → sayım. Minimum alan eşiği geliştirme kümesinde 12 değer arasından en düşük MAE ile seçilmiştir (50 px).

In [8]:
from methods import BASELINE_TEMPLATE, with_changes
from evaluation import evaluate
dev = refs[refs.group == 'development']
rows = []
for a in [0, 10, 25, 50, 100, 200, 300, 400, 500, 600, 800, 1000]:
    s, _ = evaluate(D, dev, with_changes(BASELINE_TEMPLATE, min_area=a))
    rows.append(dict(min_area=a, **{k: round(v, 4) for k, v in s.items()}))
bl = pd.DataFrame(rows); bl.to_csv('baseline_min_area_dev.csv', index=False); bl

,min_area,MAE,RMSE,Bias,ExactAcc,mean_time_s
0,0,795.7,806.7216,795.7,0.0,0.0028
1,10,45.8,46.7974,45.8,0.0,0.0067
2,25,11.3,12.5340,9.1,0.0,0.0073
3,50,5.1,6.7602,-1.9,0.2,0.0069
4,100,7.2,8.3187,-7.2,0.0,0.0076
5,200,10.1,10.4642,-10.1,0.0,0.0072
6,300,10.7,10.9407,-10.7,0.0,0.0069
7,400,11.1,11.3093,-11.1,0.0,0.0066
8,500,11.2,11.4280,-11.2,0.0,0.0082
9,600,11.3,11.5109,-11.3,0.0,0.0069


In [9]:
display(Image('figures/fig_baseline_dev.png'))

## 6-10. Kontrollü deneyler (yalnızca geliştirme kümesi)

Her aşamada önceki aşamanın en iyi konfigürasyonu ebeveyn alınır ve yalnızca o aşamanın konusu değiştirilir. Sonuçlar `experiments.csv` dosyasına yazılır ve şekiller yeniden üretilir.

In [10]:
%run run_experiments.py
pd.read_csv('experiments.csv')[['Deney ID','Aşama','Deney','Count MAE','RMSE','Bias','Exact Acc','Ort. süre (s)']]

Histogram eşleme referansı: dev t029


E02 Yoğunluk: orijinal                            MAE=11.10 RMSE=11.40 Bias=-11.10


E03 Yoğunluk: contrast stretching                 MAE=9.00 RMSE=9.57 Bias=-9.00


E04 Yoğunluk: histogram eşitleme                  MAE=1.10 RMSE=1.23 Bias=+0.50


E05 Yoğunluk: CLAHE clip 2                        MAE=7.40 RMSE=7.81 Bias=-7.40


E06 Yoğunluk: CLAHE clip 4                        MAE=6.20 RMSE=6.37 Bias=-6.20


... (37 outputs left out)

E45 Min alan: 300 px                              MAE=0.70 RMSE=0.95 Bias=+0.50


E46 Min alan: 500 px                              MAE=1.50 RMSE=1.58 Bias=-0.30


E47 Min alan: 800 px                              MAE=1.70 RMSE=1.92 Bias=-0.70
  -> S6 Min alan en iyi: E45


,Deney ID,Aşama,Deney,Count MAE,RMSE,Bias,Exact Acc,Ort. süre (s)
0,E01,S0 Baseline,Baseline (Otsu + CCL),5.1,6.760,-1.9,0.2,0.0067
1,E02,S1 Yoğunluk,Yoğunluk: orijinal,11.1,11.397,-11.1,0.0,0.0395
2,E03,S1 Yoğunluk,Yoğunluk: contrast stretching,9.0,9.571,-9.0,0.0,0.0493
3,E04,S1 Yoğunluk,Yoğunluk: histogram eşitleme,1.1,1.225,0.5,0.1,0.0396
4,E05,S1 Yoğunluk,Yoğunluk: CLAHE clip 2,7.4,7.810,-7.4,0.0,0.0427
5,E06,S1 Yoğunluk,Yoğunluk: CLAHE clip 4,6.2,6.372,-6.2,0.0,0.0424
6,E07,S1 Yoğunluk,Yoğunluk: histogram eşleme,12.2,12.369,-12.2,0.0,0.0475
7,E08,S2 Filtre,Filtre: yok,1.7,1.817,-0.3,0.1,0.0404
8,E09,S2 Filtre,Filtre: Gaussian 3x3,1.7,2.588,1.3,0.2,0.0405
9,E10,S2 Filtre,Filtre: Gaussian 5x5,1.1,1.225,0.5,0.1,0.0408


In [11]:
for f in ['fig2_intensity', 'fig_filters', 'fig3a_segmentation', 'fig3b_morphology', 'fig_parameter_effects']:
    display(Image(f'figures/{f}.png'))

### 10.6. Bütün geliştirme görüntülerinin görsel incelemesi

Baseline ve üç aday pipeline'ın 10 geliştirme karesindeki çıktıları çizilir. İnceleme notları `dev_visual_review.md` dosyasındadır.

In [12]:
from figures_dev_check import fig_dev_all
cfgs = json.load(open('experiment_configs.json'))
dev_frames = refs[refs.group == 'development']['frame'].tolist()
for n, e in [('Baseline', 'E01'), ('A', 'E45'), ('B', 'E41'), ('C', 'E21')]:
    fig_dev_all(D, dev_frames, cfgs[e], f'Geliştirme kareleri, {n} ({e})', f'figures/fig_dev_check_{n}.png')
    display(Image(f'figures/fig_dev_check_{n}.png'))

## 11. Aday pipeline'ların doğrulanması ve nihai seçim

Seçim kuralı (Bölüm 11): en düşük doğrulama MAE; eşitlikte doğrulama RMSE, ortalama işlem süresi ve pipeline karmaşıklığı (işlem adımı sayısı). Baseline yöntemi de bu aşamada `final_pipeline.json` içinde sabitlenir.

In [13]:
%run run_validation.py

Pipeline Deney ID  Aday                          Segment.  min_area  Dev MAE  Val MAE  Val RMSE  Val Bias  Val Exact Acc  Ort. süre (s)  n_steps  Seçildi
Baseline      E01 False                              Otsu        50      5.1      3.2     5.138      -2.8            0.2         0.0062        1    False
       A      E45  True                              Otsu       300      0.7      1.2     1.265       0.8            0.0         0.0381        5     True
       B      E41  True                              Otsu       300      0.8      1.2     1.265       0.8            0.0         0.2080        6    False
       C      E21  True Adaptif gaussian (blok=51, C=-20)       300      1.1      2.2     2.490       2.2            0.2         0.0452        5    False

        A   B  Baseline   C  reference_count
frame                                       
12     25  25        12  26               23
21     23  23        21  24               24
62     22  22        22  24               21
70  

In [14]:
from figures_validation import fig_validation
fp = json.load(open('final_pipeline.json'))
fig_validation(D, [12, 21, 62, 70, 72], fp['config'], 'Doğrulama kareleri, Pipeline A (E45)', 'figures/fig4_validation.png')
display(Image('figures/fig4_validation.png'))

## 12. Nihai pipeline'ın kilitlenmesi ve test tahminleri

Nihai konfigürasyon `final_pipeline.json` dosyasından okunur ve SHA-256 özeti alınır. Test görüntüleri işlenir, tahminler `test_predictions_locked.csv` dosyasına yazılır. Bu hücre test referanslarını okumaz. Tahminler ayrı bir commit ile kaydedilmiştir; test değerlendirmesi (Bölüm 13) bu commit'ten sonra yapılmıştır.

In [2]:
%run run_test_predictions.py

block  frame    filename  baseline_predicted_count  final_predicted_count
   B1     17 02/t017.tif                        15                     23
   B1     18 02/t018.tif                        17                     23
   B1     19 02/t019.tif                        25                     23
   B1     20 02/t020.tif                        18                     23
   B1     21 02/t021.tif                        22                     22
   B2     35 02/t035.tif                        21                     23
   B2     36 02/t036.tif                        20                     23
   B2     37 02/t037.tif                        23                     23
   B2     38 02/t038.tif                        23                     24
   B2     39 02/t039.tif                        22                     24
   B3     72 02/t072.tif                        17                     27
   B3     73 02/t073.tif                        20                     27
   B3     74 02/t074.tif              

## 13. Bağımsız test performansı

Kilitli dosyadaki konfigürasyon özetleri `final_pipeline.json` ile karşılaştırılarak doğrulanır. Test referans sayıları yalnızca bu aşamada okunur. Kilitli tahminler değiştirilmez ve yeniden hesaplanmaz. Sonuçlar `test_results.csv` dosyasına yazılır.

In [2]:
%run run_test_evaluation.py

Kilitli konfigürasyon özetleri doğrulandı.


block  frame  reference_count  baseline_predicted_count  final_predicted_count  baseline_abs_error  final_abs_error
   B1     17               24                        15                     23                   9                1
   B1     18               24                        17                     23                   7                1
   B1     19               25                        25                     23                   0                2
   B1     20               25                        18                     23                   7                2
   B1     21               24                        22                     22                   2                2
   B2     35               26                        21                     23                   5                3
   B2     36               27                        20                     23                   7                4
   B2     37               27                        23                 

## 14. Ardışık görüntülerde zamansal analiz ve zorunlu şekiller

Şekil 1 (akış diyagramı) ve Şekil 5 (zamansal grafik) veri seti görüntüsü içermediği için repoda bulunur. Şekil 4 ve Şekil 6 veri seti görüntülerinden üretildiği için repoya yüklenmez.

In [3]:
from figures_test import fig_temporal, fig_flowchart, fig_stages, fig_examples
fp = json.load(open('final_pipeline.json'))
fig_flowchart('figures/fig1_flowchart.png')
fig_stages(D, '01', 52, fp['config'], 'figures/fig4_stages.png')
fig_temporal('test_results.csv', 'figures/fig5_temporal.png')
items = [('(a) Başarılı: en düşük hata', '02', 17, None),
         ('(b) Hatalı: en yüksek hata', '02', 75, None),
         ('(c) Zorlayıcı: bölünen hücre ve düzensiz soluk çekirdek', '02', 35, (0, 350, 250, 900)),
         ('(d) Zorlayıcı: çok soluk çekirdekler', '02', 72, (0, 480, 0, 420))]
fig_examples(D, items, fp['config'], fp['baseline']['config'], 'figures/fig6_examples.png')
for f in ['fig1_flowchart', 'fig4_stages', 'fig5_temporal', 'fig6_examples']:
    display(Image(f'figures/{f}.png'))

In [4]:
tr = pd.read_csv('test_results.csv'); fr = tr[tr.row_type == 'frame']
rows = []
for b, g in fr.groupby('block'):
    g = g.sort_values('frame')
    for col, name in [('reference_count', 'Gerçek'), ('baseline_predicted_count', 'Baseline'),
                      ('final_predicted_count', 'Nihai')]:
        d = g[col].diff().dropna()
        rows.append(dict(blok=b, seri=name, sayımlar=g[col].astype(int).tolist(),
                         ardışık_değişimler=d.astype(int).tolist(),
                         ort_mutlak_değişim=round(d.abs().mean(), 2), en_büyük_sıçrama=int(d.abs().max())))
temporal = pd.DataFrame(rows); temporal.to_csv('temporal_analysis.csv', index=False); temporal

,blok,seri,sayımlar,ardışık_değişimler,ort_mutlak_değişim,en_büyük_sıçrama
0,B1,Gerçek,"[24, 24, 25, 25, 24]","[0, 1, 0, -1]",0.50,1
1,B1,Baseline,"[15, 17, 25, 18, 22]","[2, 8, -7, 4]",5.25,8
2,B1,Nihai,"[23, 23, 23, 23, 22]","[0, 0, 0, -1]",0.25,1
3,B2,Gerçek,"[26, 27, 27, 28, 28]","[1, 0, 1, 0]",0.50,1
4,B2,Baseline,"[21, 20, 23, 23, 22]","[-1, 3, 0, -1]",1.25,3
5,B2,Nihai,"[23, 23, 23, 24, 24]","[0, 0, 1, 0]",0.25,1
6,B3,Gerçek,"[30, 30, 30, 31, 31]","[0, 0, 1, 0]",0.25,1
7,B3,Baseline,"[17, 20, 18, 16, 21]","[3, -2, -2, 5]",3.00,5
8,B3,Nihai,"[27, 27, 27, 27, 28]","[0, 0, 0, 1]",0.25,1


## 15. Hata ayrıştırması (yalnızca analiz)

Nihai pipeline değiştirilmeden, her referans hücre işaretinin hangi tahmin nesnesiyle örtüştüğü incelenir. Kaçırılan hücre, hiçbir nesneyle örtüşmeyen referanstır; birleşmiş nesne, birden fazla referans içeren nesnedir; sahte nesne, hiçbir referans içermeyen nesnedir. Sonuçlar `error_analysis.csv` dosyasına yazılır.

In [2]:
from error_analysis import decompose_set
fp = json.load(open('final_pipeline.json'))
ea = pd.concat([decompose_set(D, [12, 21, 62, 70, 72], fp['config'], '01').assign(group='validation'),
                decompose_set(D, [17, 18, 19, 20, 21, 35, 36, 37, 38, 39, 72, 73, 74, 75, 76], fp['config'], '02').assign(group='test')])
ea.to_csv('error_analysis.csv', index=False)
ea.groupby('group')[['reference', 'predicted', 'missed', 'merged_lost', 'false_positive']].sum()

,reference,predicted,missed,merged_lost,false_positive
group,,,,,
test,410,367,50,0,7
validation,110,114,3,0,7


In [3]:
# Şekil 1, 4 ve 6 düzeltilmiş etiketlerle yeniden üretilir (pipeline ve tahminler değişmez).
from figures_test import fig_flowchart, fig_stages, fig_examples
fig_flowchart('figures/fig1_flowchart.png')
fig_stages(D, '01', 52, fp['config'], 'figures/fig4_stages.png')
items = [('(a) Başarılı: en düşük hata', '02', 17, None),
         ('(b) Hatalı: en yüksek hata', '02', 75, None),
         ('(c) Zorlayıcı: bölünen hücre ve düzensiz soluk çekirdek', '02', 35, (0, 350, 250, 900)),
         ('(d) Zorlayıcı: çok soluk çekirdekler', '02', 72, (0, 480, 0, 420))]
fig_examples(D, items, fp['config'], fp['baseline']['config'], 'figures/fig6_examples.png')

## 16. Rapor şekilleri

Raporda sayfa yerini azaltmak için Şekil 2, 3 ve 6'nın kompakt sürümleri kullanılmıştır. Yöntem ve sonuçlar değişmez. Bu şekiller veri seti görüntüsü içerdiği için repoya yüklenmez.

In [2]:
from experiments import choose_hist_match_reference
from figures_report import fig2_compact, fig3_compact, fig6_compact
choose_hist_match_reference(D, refs[refs.group == 'development'] if 'refs' in dir() else pd.read_csv('dev_val_references.csv', dtype={'sequence': str}).query("group == 'development'"))
fp = json.load(open('final_pipeline.json'))
fig2_compact(D, 52, ['E02', 'E03', 'E04', 'E05', 'E06', 'E07'], 'figures/report_fig2.png')
fig3_compact(D, 52, ['E16', 'E18', 'E21', 'E24', 'E25'], ['E26', 'E29', 'E31', 'E34', 'E36'], 'figures/report_fig3.png')
fig6_compact(D, [('(a) Başarılı', '02', 17, None), ('(b) Hatalı', '02', 75, None),
                 ('(c) Zorlayıcı: bölünen hücre', '02', 35, (0, 440, 440, 900)),
                 ('(d) Zorlayıcı: çok soluk çekirdekler', '02', 72, (0, 440, 0, 440))],
             fp['config'], fp['baseline']['config'], 'figures/report_fig6.png')